Первый вариант оформления теста Хаусмана в виде функции.
Для более полной реализации см. файл **ClassHausmanTest.ipynb**

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from linearmodels.panel import PanelOLS, RandomEffects

In [4]:


def hausman_test(df, y_var, x_vars):
    y = df[y_var]
    X = df[x_vars]
    X = sm.add_constant(X)
    
    fe_model = PanelOLS(y, X, entity_effects=True)
    fe_res = fe_model.fit()

    re_model = RandomEffects(y, X)
    re_res = re_model.fit()

    b_FE = fe_res.params
    b_RE = re_res.params

    common_coef = b_FE.index.intersection(b_RE.index)

    b_FE = b_FE[common_coef]
    b_RE = b_RE[common_coef]

    V_FE = fe_res.cov.loc[common_coef, common_coef]
    V_RE = re_res.cov.loc[common_coef, common_coef]

    diff = b_FE - b_RE

    #сама статистика Хаусмана по формуле
    stat = np.dot(np.dot(diff.T, np.linalg.inv(V_FE - V_RE)),diff)

    df_h = len(diff)

    p_value = 1 - stats.chi2.cdf(stat, df_h)

    return pd.DataFrame({
        "Статистика Хаусмана":[stat],
        "df":[df_h],
        "p_value":[p_value]
    })